# 01 — La fonction ζ(s) : naissance d'une série

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.special import zeta as scipy_zeta
from plotly.subplots import make_subplots

TEMPLATE     = "plotly_dark"
COLOR_MAIN   = "#7C6FCD"
COLOR_ACCENT = "#F2A623"
COLOR_MUTED  = "#888780"
COLOR_DANGER = "#E24B4A"

## ζ(2) — le problème de Bâle

La série $\zeta(2) = \sum_{n=1}^{\infty} \frac{1}{n^2} = 1 + \frac{1}{4} + \frac{1}{9} + \cdots$ converge vers une valeur finie.

Euler (1734) montre que cette valeur est exactement $\frac{\pi^2}{6} \approx 1.6449$ — une connexion inattendue entre une somme de fractions et le cercle.

La convergence est **lente** : avec 100 000 termes, l'erreur est encore ~1e-5.  
Il reste une infinité de termes ignorés qui s'accumulent.

In [2]:
# Ce qu'on a construit : approximation de ζ(2) par somme partielle
n = np.arange(1, 100_000)
approx = np.sum(1 / n**2)
exact  = np.pi**2 / 6

print(f"Somme partielle (N=100 000) : {approx:.10f}")
print(f"Valeur exacte π²/6          : {exact:.10f}")
print(f"Erreur                      : {exact - approx:.2e}")

Somme partielle (N=100 000) : 1.6449240668
Valeur exacte π²/6          : 1.6449340668
Erreur                      : 1.00e-05


## Convergence interactive — faire grandir N

Le slider permet de faire varier N et voir l'erreur diminuer.  
On visualise à la fois la somme partielle et l'écart avec π²/6.

In [7]:
N_values  = [10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
errors    = []
approxs   = []
exact     = np.pi**2 / 6

for N in N_values:
    n   = np.arange(1, N + 1, dtype=float)
    val = np.sum(1 / n**2)
    approxs.append(val)
    errors.append(exact - val)


fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Somme partielle vs π²/6", "Erreur absolue (échelle log)"))

fig.add_trace(go.Scatter(
    x=N_values, y=approxs,
    name="Somme partielle",
    mode="lines+markers",
    line=dict(color=COLOR_MAIN, width=2),
    marker=dict(size=7)
), row=1, col=1)

fig.add_hline(y=exact, line_dash="dot", line_color=COLOR_ACCENT,
              annotation_text="π²/6", annotation_font_color=COLOR_ACCENT,
              row=1, col=1)

fig.add_trace(go.Scatter(
    x=N_values, y=errors,
    name="Erreur",
    mode="lines+markers",
    line=dict(color=COLOR_DANGER, width=2),
    marker=dict(size=7),
    hovertemplate="N=%{x}<br>erreur=%{y:.2e}<extra></extra>"
), row=1, col=2)

fig.update_xaxes(type="log", title_text="N (nombre de termes)")
fig.update_yaxes(type="log", title_text="Erreur", row=1, col=2)
fig.update_yaxes(title_text="Valeur", row=1, col=1)

fig.update_layout(template=TEMPLATE, height=420,
    title="ζ(2) — convergence vers π²/6 en fonction de N")
fig.show()

## Généralisation : ζ(s) pour tout s réel > 1

$$\zeta(s) = \sum_{n=1}^{\infty} \frac{1}{n^s}$$

On generalise : au lieu de fixer $s=2$, on passe $s$ en paramètre.  
Plus $s$ est grand, plus les termes $1/n^s$ décroissent vite → convergence rapide.  
Plus $s$ est proche de 1, plus la convergence est lente.

In [4]:
def zeta_partial_sum(s, N):
    """Somme partielle de ζ(s) = Σ 1/n^s pour n de 1 à N."""
    n = np.arange(1, N + 1, dtype=float)
    return np.cumsum(1.0 / n**s)  # cumsum : on garde l'historique terme par terme

# Test sur quelques valeurs
N_MAX = 500
ns    = np.arange(1, N_MAX + 1)

s_values = [1.5, 2.0, 3.0, 5.0]
colors   = [COLOR_DANGER, COLOR_MAIN, COLOR_ACCENT, "#1D9E75"]

fig = go.Figure()

for s, color in zip(s_values, colors):
    cumsum = zeta_partial_sum(s, N_MAX)
    exact  = float(scipy_zeta(s))

    fig.add_trace(go.Scatter(
        x=ns, y=cumsum,
        name=f"s = {s}",
        line=dict(color=color, width=2),
        hovertemplate=f"N=%{{x}}<br>ζ({s})≈%{{y:.5f}}<extra></extra>"
    ))
    fig.add_hline(y=exact, line_dash="dot", line_color=color, opacity=0.4,
                  annotation_text=f"ζ({s})={exact:.4f}",
                  annotation_font_color=color)

fig.update_layout(
    template=TEMPLATE,
    title="Convergence de ζ(s) pour différentes valeurs de s",
    xaxis_title="N (nombre de termes)",
    yaxis_title="Somme partielle",
    height=460,
    hovermode="x unified"
)
fig.show()

## La singularité en s = 1 — la série harmonique

Pour $s = 1$, la série devient :

$$H_N = 1 + \frac{1}{2} + \frac{1}{3} + \frac{1}{4} + \cdots$$

C'est la **série harmonique** — elle **diverge**. Elle tend vers l'infini, mais d'une lenteur extrême.  
On peut montrer que $H_N \approx \ln(N) + \gamma$ où $\gamma \approx 0.5772$ (constante d'Euler-Mascheroni).

Conséquence : $\zeta(s)$ n'est **pas définie** en $s=1$ par la série.  
C'est le seul point singulier — un **pôle**. On y reviendra dans le notebook 02.

In [9]:
N_MAX = 10000
ns    = np.arange(1, N_MAX + 1, dtype=float)

harmonic = np.cumsum(1.0 / ns)           # s = 1 : diverge
log_fit  = np.log(ns) + 0.5772           # approximation ln(N) + γ
s15      = np.cumsum(1.0 / ns**1.5)      # s = 1.5 : converge

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ns, y=harmonic,
    name="s = 1 (harmonique, diverge)",
    line=dict(color=COLOR_DANGER, width=2)
))
fig.add_trace(go.Scatter(
    x=ns, y=log_fit,
    name="ln(N) + γ  (approximation)",
    line=dict(color=COLOR_MUTED, width=1.5, dash="dash")
))
fig.add_trace(go.Scatter(
    x=ns, y=s15,
    name="s = 1.5 (converge)",
    line=dict(color=COLOR_MAIN, width=2)
))
fig.add_hline(
    y=float(scipy_zeta(1.5)),
    line_dash="dot", line_color=COLOR_MAIN, opacity=0.5,
    annotation_text=f"ζ(1.5)={scipy_zeta(1.5):.4f}",
    annotation_font_color=COLOR_MAIN
)

fig.update_layout(
    template=TEMPLATE,
    title="s=1 : divergence harmonique vs s=1.5 : convergence",
    xaxis_title="N",
    yaxis_title="Somme partielle",
    height=430,
    hovermode="x unified"
)
fig.show()

print(f"H(10 000)    = {harmonic[-1]:.4f}")
print(f"ln(10000)+γ  = {np.log(10000)+0.5772:.4f} approximation")
print(f"ζ(1.5)       = {float(scipy_zeta(1.5)):.6f} converge ")

H(10 000)    = 9.7876
ln(10000)+γ  = 9.7875 approximation
ζ(1.5)       = 2.612375 converge 
